# Logistic regression

Here, we are considering a simple problem of classifying data into 2 different categories - Malignant vs Benign tumors for given input data of patients. This type of problem is also called binary classification because there are only 2 classes.

For concreteness, we consider the following dataset for the problem:

[Breast cancer dataset](https://huggingface.co/datasets/mnemoraorg/wisconsin-breast-cancer-diagnostic)

The dataset contains following fields:
1. radius: The mean of distances from the center to points on the perimeter.
2. texture: The standard deviation of grayscale values within the region of interest.
3. perimeter: The perimeter of the cell nuclei's contour.
4. area: The area of the cell nuclei's contour.
5. smoothness: The local variation in radius lengths.
6. compactness: Defined as (perimeter^2 / area - 1.0).
7. concavity: The severity of concave portions of the contour.
8. concave points: The number of concave portions of the contour.
9. symmetry: The symmetry of the cell nuclei.
10. fractal_dimension: The "coastline approximation" of the cell nuclei's boundary, calculated as 1 - fractal dimension.


In [ ]:
import numpy as np
import pandas as pd

import torch as t
from torch import nn
from torch.nn import Module
from torch.optim.adamw import AdamW
from torch.utils.data import DataLoader

from tqdm import tqdm

from datasets import load_dataset, Dataset
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

# Site color palette (matches the blog theme)
SITE = dict(
    bg_primary   = '#213636',
    bg_secondary = '#1a2d2d',
    bg_tertiary  = '#2a4444',
    border       = '#3a5454',
    text_primary = '#c8c4b8',
    text_secondary = '#a09890',
    accent       = '#bdb76b',
    olive        = '#828631',
)

PLOTS_DIR = Path("./plots")
PLOTS_DIR.mkdir(exist_ok=True)

## Pre-process the dataset

We pull data from hugging face and also split it into train:test ratio of 10:1.
There are some columns which leak information and can lead to model learning spurious signals.

For example:
`id` (unique identifier), `diagnosis` (label), `unnamed` (empty, not required)

In [16]:
ds = load_dataset("mnemoraorg/wisconsin-breast-cancer-diagnostic", split="train")
def map_label(x):
    if x["diagnosis"] == "M":
        x["diagnosis"] = 1
    else:
        x["diagnosis"] = 0
    return x

ds = ds.remove_columns(["id", "Unnamed: 32"])
ds = ds.train_test_split(test_size=0.1).map(map_label)

# Normalize features: compute stats on the TRAIN split only (no test leakage),
# then apply them to both splits via a batched map.
feature_cols = [c for c in ds["train"].column_names if c != "diagnosis"]
train_matrix = ds["train"].to_pandas()[feature_cols].to_numpy(dtype=np.float64)
mean = train_matrix.mean(axis=0)
std = train_matrix.std(axis=0)
std[std == 0] = 1.0  # guard against constant features

def normalize(batch: dict) -> dict:
    vals = np.column_stack([np.array(batch[c], dtype=np.float64) for c in feature_cols])
    normed = (vals - mean) / std
    return {f"norm_{c}": normed[:, i] for i, c in enumerate(feature_cols)}

ds = ds.map(normalize, batched=True, load_from_cache_file=False)

Map: 100%|██████████| 57/57 [00:00<00:00, 7069.89 examples/s]


## Logistic regression model

Logistic regression model has following mathematical relationship:

$$
h_{\theta}(z) = \frac{1}{1 + \exp(-z)}
$$

where, 
$$
z = \theta^T x \qquad \text{where, } x \in \R^n
$$

Also,

$$
\frac{\partial h_{\theta}}{\partial z} = \frac{\partial}{\partial z} \frac{1}{1+\exp(-z)}\\
= \frac{- \frac{\partial}{\partial z} (1+\exp(-z))}{(1+\exp(-z))^2}\\
= \frac{\exp(-z)}{(1+\exp(-z))^2}\\
= \frac{1}{1+\exp(-z)} \frac{\exp(-z)}{1+\exp(-z)}\\
= h_{\theta} (1 - h_{\theta})
$$

Now, 
$$
\frac{\partial h_{\theta}}{\partial \theta} = \frac{\partial h_{\theta}}{\partial z} \frac{\partial z}{\partial \theta}\\
= h_{\theta} (1 - h_{\theta}) \frac{\partial \theta^T x}{\partial \theta}\\
= h_{\theta} (1 - h_{\theta}) x\\
\boxed{\frac{\partial h_{\theta}}{\partial \theta} = h_{\theta} (1 - h_{\theta}) x}
$$

In [17]:
## Why not linear regression? (a picture is worth a thousand words)

# We fit ordinary least squares on a tiny 1-D toy dataset and compare it
# with logistic regression. Two failure modes of the linear fit show up:
#   1. Predictions escape [0, 1] - they are not probabilities.
#   2. A single far-away outlier drags the fitted line and shifts the
#      0.5-threshold (the decision boundary) substantially.

def compare_linear_vs_logistic():
    # Toy data: tumor size vs malignant (1) / benign (0)
    x_toy = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20], dtype=np.float64)  # 20 = outlier
    y_toy = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], dtype=np.float64)

    # Ordinary least squares (closed form): y = a*x + c
    a, c = np.polyfit(x_toy, y_toy, 1)

    # Logistic regression via gradient descent on the same data
    W_lr = t.randn(1, 1, requires_grad=True)
    b_lr = t.randn(1, requires_grad=True)
    x_t = t.tensor(x_toy, dtype=t.float32).view(-1, 1)
    y_t = t.tensor(y_toy, dtype=t.float32).view(-1, 1)
    opt = AdamW([W_lr, b_lr], lr=0.1)
    for _ in range(2000):
        h = t.nn.functional.sigmoid(x_t @ W_lr + b_lr)
        loss = t.nn.functional.binary_cross_entropy(h, y_t)
        opt.zero_grad()
        loss.backward()
        opt.step()

    xs = np.linspace(0, 21, 200)
    ols_line = a * xs + c
    logit_curve = t.nn.functional.sigmoid(
        t.tensor(xs, dtype=t.float32).view(-1, 1) @ W_lr + b_lr
    ).detach().numpy().squeeze()

    # Where each method crosses 0.5
    ols_boundary = (0.5 - c) / a
    logit_boundary = (-b_lr / W_lr).item()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_toy, y=y_toy, mode="markers", name="Data",
        marker=dict(color=[SITE['accent'] if v == 1 else SITE['olive'] for v in y_toy], size=9),
    ))
    fig.add_trace(go.Scatter(
        x=xs, y=ols_line, mode="lines", name="Linear regression fit",
        line=dict(color="#e07a5f", width=2),
    ))
    fig.add_trace(go.Scatter(
        x=xs, y=logit_curve, mode="lines", name="Logistic regression fit",
        line=dict(color="#81b29a", width=2),
    ))
    fig.add_hline(y=0.5, line_dash="dot", line_color=SITE['text_secondary'],
                  annotation_text="threshold = 0.5", annotation_font_color=SITE['text_secondary'])
    fig.add_vrect(x0=ols_boundary, x1=ols_boundary, line_width=0)
    fig.add_trace(go.Scatter(
        x=[ols_boundary], y=[0.5], mode="markers", name="OLS decision boundary",
        marker=dict(color="#e07a5f", size=14, symbol="diamond"),
    ))
    fig.add_trace(go.Scatter(
        x=[logit_boundary], y=[0.5], mode="markers", name="Logistic decision boundary",
        marker=dict(color="#81b29a", size=14, symbol="diamond"),
    ))
    fig.update_layout(
        title=dict(text="Linear vs logistic regression on a classification toy problem",
                   font=dict(color=SITE['text_primary'])),
        xaxis=dict(title="tumor size (arbitrary units)", color=SITE['text_secondary'],
                   gridcolor=SITE['border']),
        yaxis=dict(title="predicted value / probability", color=SITE['text_secondary'],
                   gridcolor=SITE['border'], range=[-0.3, 1.3]),
        paper_bgcolor=SITE['bg_secondary'],
        plot_bgcolor=SITE['bg_primary'],
        font=dict(color=SITE['text_primary']),
        legend=dict(orientation="h", x=0.5, y=-0.2, xanchor="center", yanchor="top",
                    bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=60),
    )
    fig.write_html(
        PLOTS_DIR / "logistic_regression_vs_linear.html",
        config=dict(responsive=True, displayModeBar=True),
        include_plotlyjs='cdn',
    )
    fig.show()

    print(f"OLS fit: y = {a:.3f}x + {c:.3f}")
    print(f"OLS predictions escape [0,1] for x < {(0-c)/a:.1f} and x > {(1-c)/a:.1f}")
    print(f"OLS decision boundary (y=0.5): x = {ols_boundary:.2f}")
    print(f"Logistic decision boundary (p=0.5): x = {logit_boundary:.2f}")


compare_linear_vs_logistic()

OLS fit: y = 0.070x + 0.070
OLS predictions escape [0,1] for x < -1.0 and x > 13.3
OLS decision boundary (y=0.5): x = 6.17
Logistic decision boundary (p=0.5): x = 5.47


In [18]:
def plot_sigmoid_with_derivative():
    x = t.linspace(-5, 5, steps=100)
    y = nn.functional.sigmoid(x)
    dy_dx = y*(1-y)

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True)
    fig.add_trace(
             go.Scatter(x=x, y=y, mode='lines', name='Sigmoid',
                        line=dict(color=SITE['accent'])),
             row=1,
             col=1
        )
    fig.add_trace(
        go.Scatter(x=x, y=dy_dx, mode='lines', name='Sigmoid derivative',
                    line=dict(color=SITE['olive'])),
        row=2,
        col=1
    )
    fig.update_layout(
            title=dict(text='Sigmoid and its derivative', font=dict(color=SITE['text_primary'])),
            paper_bgcolor=SITE['bg_secondary'],
            plot_bgcolor=SITE['bg_primary'],
            font=dict(color=SITE['text_primary']),
            legend=dict(orientation='h', x=0.5, y=-0.15, xanchor='center', yanchor='top',
                        bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
            autosize=True,
            margin=dict(l=60, r=40, t=80, b=60),
        )
    for axis in ['xaxis', 'yaxis', 'xaxis2', 'yaxis2']:
        fig.update_layout({axis: dict(gridcolor=SITE['border'], color=SITE['text_secondary'])})
    fig.write_html(
        PLOTS_DIR / "sigmoid_and_its_derivative.html",
        config=dict(responsive=True, displayModeBar=True),
        include_plotlyjs='cdn',
    )
    fig.show()

plot_sigmoid_with_derivative()

In [19]:
class LogisticRegression(Module):

    def __init__(self, size):
        super().__init__()
        self.W = nn.Parameter(t.randn(size=size))
        self.b = nn.Parameter(t.randn(size=(1,)))

    def forward(self, x: t.Tensor) -> t.Tensor:
        z = x @ self.W + self.b
        return nn.functional.sigmoid(z)


def test_model(model: nn.Module, input: t.Tensor):
    W = model.W
    b = model.b
    z = input @ W + b
    expected = nn.functional.sigmoid(z)
    actual = model(input)
    assert t.allclose(expected, actual), f"{expected} != {actual}"

x = t.randn(size=(1, 3))
m = LogisticRegression(size=(3, 1))

test_model(m, x)

## Training the model

### Maximum likelihood Estimation connection

To see the connection with MLE, we start from first principles. 
If we think of tumor as some form of a random variable $y$ that distributes over a population of patients getting diagnosed, 
then we may say that
$$
p(y|x;\theta) = \left\{
\begin{array}{ll}
h_{\theta}(x) & \text{if } y = 1 \\
1 - h_{\theta}(x) & \text{if } y = 0
\end{array}
\right.
$$

$y$ is dependent upon the features $x_i$ of the patients that we collected in our dataset of size $N$.

We can write it more concisely as [Bernoulli data distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution).
$$
p(y_i | x_i; \theta) = h_{\theta}(x_i)^{y_i} (1 - h_{\theta}(x_i))^{1-y_i}
$$

Now, again, connecting with MLE, we see that
$$
L(\theta) = p(y_1, y_2, y_3, ..., y_N | x_1, x_2, x_3, ..., x_N; \theta)\\
= \prod_{i=1}^N p(y_i | x_i; \theta) \qquad \text{since i.i.d.}\\
= \prod_{i=1}^N h_{\theta}(x_i)^{y_i} (1 - h_{\theta}(x_i))^{1-y_i}\\
$$

Taking $\log$ both sides we get

$$
\log L(\theta) = \log \prod_{i=1}^N h_{\theta}(x_i)^{y_i} (1 - h_{\theta}(x_i))^{1-y_i}\\
= \sum_{i=1}^N \log h_{\theta}(x_i)^{y_i} (1 - h_{\theta}(x_i))^{1-y_i}\\
= \sum_{i=1}^N y_i \log h_{\theta}(x_i) + (1 - y_i) \log (1 - h_{\theta}(x_i))
$$

For brevity, let $h_i = h_{\theta}(x_i)$. Differentiating both sides with $\theta_j$ and setting to $0$ gives us:

$$
\frac{\partial}{\partial \theta_j} \log L(\theta) = \frac{\partial}{\partial \theta_j} \sum_{i=1}^N y_i \log h_i + (1 - y_i) \log (1 - h_i)\\
= \sum_{i=1}^N \left(\frac{y_i}{h_i} - \frac{1 - y_i}{1 - h_i}\right) \frac{\partial h_i}{\partial \theta_j}\\
= \sum_{i=1}^N \left(\frac{y_i}{h_i} - \frac{1 - y_i}{1 - h_i}\right) \left(h_i \left(1 - h_i\right) x_{ij}\right)\\
= \sum_{i=1}^N \left(y_i \left(1 - h_i\right) - \left(1 - y_i\right) h_i\right) x_{ij}\\
= \sum_{i=1}^N \left(y_i - h_i\right) x_{ij}\\
\boxed{\frac{\partial}{\partial \theta} \log L(\theta) = \sum_{i=1}^N \left(y_i - h_{\theta}(x_i)\right) x_i}
$$

Note that every term in the sum contributes to the derivative w.r.t. $\theta_j$ (through the $j$-th feature $x_{ij}$ of each example), which is why the sum stays intact.

### Connection with Binary cross entropy loss

The [BCELoss](https://docs.pytorch.org/docs/2.14/generated/torch.nn.BCELoss.html) as per Pytorch docs is defined as:

$$
\text{BCE}(\theta) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log h_{\theta}(x_i) + (1 - y_i) \log (1 - h_{\theta}(x_i)) \right]\\
= -\frac{1}{N} \log L(\theta)
$$

So BCE is the negative average log-likelihood, and its gradient is:

$$
\nabla_{\theta} \text{BCE}(\theta) = -\frac{1}{N} \sum_{i=1}^N \left(y_i - h_{\theta}(x_i)\right) x_i
$$

Therefore, minimizing BCE loss is exactly the same as maximizing the log-likelihood. This is why `loss.backward()` in our training loop performs gradient *descent* on BCE, which is gradient *ascent* on $\log L(\theta)$.

In [20]:
norm_cols = [f"norm_{c}" for c in feature_cols]

def to_features(batch: dict) -> dict:
    # with_transform receives a batch (dict of lists) and returns tensors,
    # which the DataLoader then collates automatically.
    return {
        "x": t.stack([t.tensor(batch[c], dtype=t.float32) for c in norm_cols], dim=1),
        "y": t.tensor(batch["diagnosis"], dtype=t.float32),
    }

def is_match(predicted: t.Tensor, target: t.Tensor) -> t.Tensor:
    return (predicted == target).sum().item()

def train(model: nn.Module, ds: Dataset, epochs=100, lr=1e-3, eps=1e-5):
    X_train = DataLoader(ds["train"].with_transform(to_features), batch_size=16, shuffle=True)
    X_val = DataLoader(ds["test"].with_transform(to_features), batch_size=16)
    opt = AdamW(model.parameters(), lr=lr, eps=eps)

    losses, scores = [], []
    for epoch in tqdm(range(epochs), desc="Epochs"):
        batch_losses = []
        for batch in X_train:
            target = batch["y"]
            predicted = model(batch["x"])
            loss = nn.functional.binary_cross_entropy(predicted, target.view(-1, 1))
            opt.zero_grad()
            loss.backward()
            opt.step()
            batch_losses.append(loss.item())

        mean_loss = sum(batch_losses)/len(batch_losses)
        losses.append(mean_loss)

        # Evaluate accuracy on the whole validation set
        correct, total = 0, 0
        with t.no_grad():
            for batch in X_val:
                target = batch["y"]
                predicted = model(batch["x"])
                predicted = (predicted >= 0.5).float()
                correct += is_match(predicted, target.view(-1, 1))
                total += len(target)
        mean_score = correct / total
        scores.append(mean_score)

        if (epoch+1) % 100 == 0:
            print(f"Epoch: {epoch+1} Loss: {mean_loss:.3f} Accuracy: {mean_score:.3f}")

    x = list(range(1, epochs+1))
    fig = make_subplots(rows=2, cols=1, subplot_titles=('BCE Loss', 'Accuracy'))
    fig.add_trace(
         go.Scatter(x=x, y=losses, mode='lines', name='Train loss',
                    line=dict(color=SITE['accent'])),
         row=1,
         col=1
    )
    fig.add_trace(
        go.Scatter(x=x, y=scores, mode='lines', name='Accuracy',
                   line=dict(color=SITE['olive'])),
        row=2,
        col=1
    )
    fig.update_layout(
        title=dict(text='Training Progress', font=dict(color=SITE['text_primary'])),
        paper_bgcolor=SITE['bg_secondary'],
        plot_bgcolor=SITE['bg_primary'],
        font=dict(color=SITE['text_primary']),
        legend=dict(orientation='h', x=0.5, y=-0.15, xanchor='center', yanchor='top',
                    bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
        autosize=True,
        margin=dict(l=60, r=40, t=80, b=60),
    )
    for axis in ['xaxis', 'yaxis', 'xaxis2', 'yaxis2']:
        fig.update_layout({axis: dict(gridcolor=SITE['border'], color=SITE['text_secondary'])})
    fig.write_html(
        PLOTS_DIR / "logistic_regression_loss_curves.html",
        config=dict(responsive=True, displayModeBar=True),
        include_plotlyjs='cdn',
    )
    fig.show()


model = LogisticRegression(size=(len(feature_cols), 1))
train(model, ds, epochs=1000, lr=1e-4)

Epochs:  11%|█         | 108/1000 [00:02<00:18, 49.36it/s]

Epoch: 100 Loss: 2.216 Accuracy: 0.263


Epochs:  21%|██        | 208/1000 [00:04<00:16, 49.02it/s]

Epoch: 200 Loss: 0.632 Accuracy: 0.754


Epochs:  31%|███       | 308/1000 [00:06<00:14, 48.84it/s]

...

Epoch: 800 Loss: 0.066 Accuracy: 0.965


Epochs:  91%|█████████ | 908/1000 [00:19<00:01, 47.45it/s]

Epoch: 900 Loss: 0.063 Accuracy: 0.965


Epochs: 100%|██████████| 1000/1000 [00:21<00:00, 47.04it/s]

Epoch: 1000 Loss: 0.061 Accuracy: 0.965


In [24]:
## Visualizing the decision boundary

# The full boundary lives in 30-D space and cannot be plotted directly.
# Instead we project the (normalized) features onto their top-2 principal
# components. Since the model is linear, its boundary stays a straight
# line in this 2-D view.

# 1. Collect the normalized training data as a matrix
X_all = np.column_stack([np.array(ds["train"][c], dtype=np.float64) for c in norm_cols])
y_all = np.array(ds["train"]["diagnosis"])

# 2. PCA: top-2 principal components
centered = X_all - X_all.mean(axis=0)
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
P = Vt[:2].T                      # (n_features, 2) projection matrix
X2d = centered @ P                # data in 2-D

# 3. Grid over the 2-D PCA space, mapped back to feature space
pad = 0.5
gx, gy = np.meshgrid(
    np.linspace(X2d[:, 0].min() - pad, X2d[:, 0].max() + pad, 200),
    np.linspace(X2d[:, 1].min() - pad, X2d[:, 1].max() + pad, 200),
)
grid2d = np.column_stack([gx.ravel(), gy.ravel()])
grid_nd = grid2d @ P.T + X_all.mean(axis=0)   # invert PCA back to 30-D

# 4. Model probabilities on the grid
with t.no_grad():
    probs = model(t.tensor(grid_nd, dtype=t.float32)).numpy().reshape(gx.shape)

# 5. Plot: probability contour + decision boundary (p=0.5) + data points
# Site colorscale: bg_secondary → bg_tertiary → border → olive → accent
SITE_COLORSCALE = [
    [0.0,  '#1a2d2d'],
    [0.25, '#2a4444'],
    [0.5,  '#3a5454'],
    [0.75, '#828631'],
    [1.0,  '#bdb76b'],
]

fig = go.Figure()
fig.add_trace(go.Contour(
    x=gx[0], y=gy[:, 0], z=probs,
    colorscale=SITE_COLORSCALE, opacity=0.75,
    showscale=True,
    colorbar=dict(
        title=dict(text='p(Malignant)', font=dict(color=SITE['text_primary'])),
        tickfont=dict(color=SITE['text_secondary']),
        bgcolor=SITE['bg_secondary'],
        bordercolor=SITE['border'],
    ),
))
fig.add_trace(go.Scatter(
    x=X2d[y_all == 0, 0], y=X2d[y_all == 0, 1],
    mode="markers", name="Benign",
    marker=dict(color="#81b29a", size=6, opacity=0.8,
                line=dict(color=SITE['bg_secondary'], width=1)),
))
fig.add_trace(go.Scatter(
    x=X2d[y_all == 1, 0], y=X2d[y_all == 1, 1],
    mode="markers", name="Malignant",
    marker=dict(color="#e07a5f", size=6, opacity=0.8,
                line=dict(color=SITE['bg_secondary'], width=1)),
))
fig.update_layout(
    title=dict(text="Decision boundary in PCA space (top-2 principal components)",
               font=dict(color=SITE['text_primary'])),
    xaxis=dict(title="PC 1", color=SITE['text_secondary'], gridcolor=SITE['border']),
    yaxis=dict(title="PC 2", color=SITE['text_secondary'], gridcolor=SITE['border']),
    paper_bgcolor=SITE['bg_secondary'],
    plot_bgcolor=SITE['bg_primary'],
    font=dict(color=SITE['text_primary']),
    legend=dict(orientation="h", x=0.5, y=-0.15, xanchor="center", yanchor="top",
                bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
    autosize=True,
    margin=dict(l=60, r=40, t=60, b=60),
)
fig.write_html(
    PLOTS_DIR / "logistic_regression_decision_boundary_2d.html",
    config=dict(responsive=True, displayModeBar=True),
    include_plotlyjs='cdn',
)
fig.show()

In [25]:
## Visualizing the decision boundary in 3D (top-3 principal components)

# In 3-D PCA space the linear decision boundary becomes a flat plane.
# The model's decision function is f(x) = W^T x + b; in PCA coordinates
# u = P^T (x - mean), it becomes f(u) = (P^T W)^T u + b, still linear,
# so the boundary f(u) = 0 is a plane we can draw exactly.

# 1. PCA with top-3 components
centered3 = X_all - X_all.mean(axis=0)
U3, S3, Vt3 = np.linalg.svd(centered3, full_matrices=False)
P3 = Vt3[:3].T                      # (n_features, 3)
X3d = centered3 @ P3                # data in 3-D

# 2. Model weights expressed in PCA coordinates
W_pca = model.W.detach().numpy().T @ P3          # (1, 3)
b_pca = model.b.detach().numpy().item()

# 3. Build the decision plane f(u) = 0 over a grid of (u1, u2),
#    solving for u3: u3 = -(w1*u1 + w2*u2 + b) / w3
w1, w2, w3 = W_pca[0]
pad = 0.5
u1, u2 = np.meshgrid(
    np.linspace(X3d[:, 0].min() - pad, X3d[:, 0].max() + pad, 20),
    np.linspace(X3d[:, 1].min() - pad, X3d[:, 1].max() + pad, 20),
)
u3 = -(w1 * u1 + w2 * u2 + b_pca) / w3

# 4. Plot: decision plane + data points
fig = go.Figure()
fig.add_trace(go.Surface(
    x=u1, y=u2, z=u3,
    opacity=0.5, showscale=False,
    colorscale=[[0, SITE['border']], [1, SITE['accent']]],
    name="Decision boundary (p=0.5)",
    contours=dict(
        x=dict(show=True, color=SITE['border'], size=1.0),
        y=dict(show=True, color=SITE['border'], size=1.0),
    ),
))
fig.add_trace(go.Scatter3d(
    x=X3d[y_all == 0, 0], y=X3d[y_all == 0, 1], z=X3d[y_all == 0, 2],
    mode="markers", name="Benign",
    marker=dict(color="#81b29a", size=3, opacity=0.8),
))
fig.add_trace(go.Scatter3d(
    x=X3d[y_all == 1, 0], y=X3d[y_all == 1, 1], z=X3d[y_all == 1, 2],
    mode="markers", name="Malignant",
    marker=dict(color="#e07a5f", size=3, opacity=0.8),
))

# Shared axis style (matches the gradient descent loss-surface plot)
axis_style = dict(
    backgroundcolor=SITE['bg_tertiary'],
    gridcolor=SITE['border'],
    showbackground=True,
    zerolinecolor=SITE['border'],
    tickfont=dict(color=SITE['text_secondary']),
    title_font=dict(color=SITE['text_primary']),
)

fig.update_layout(
    title=dict(text="Decision boundary plane in PCA space (top-3 principal components)",
               font=dict(color=SITE['text_primary'])),
    paper_bgcolor=SITE['bg_secondary'],
    scene=dict(
        xaxis=dict(**axis_style, title='PC 1'),
        yaxis=dict(**axis_style, title='PC 2'),
        zaxis=dict(**axis_style, title='PC 3'),
        bgcolor=SITE['bg_primary'],
    ),
    legend=dict(
        orientation='h',
        x=0.5,
        y=-0.12,
        xanchor='center',
        yanchor='top',
        font=dict(color=SITE['text_primary']),
        bgcolor=SITE['bg_secondary'],
        bordercolor=SITE['border'],
    ),
    autosize=True,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.write_html(
    PLOTS_DIR / "logistic_regression_decision_boundary_3d.html",
    config=dict(responsive=True, displayModeBar=True),
    include_plotlyjs='cdn',
)
fig.show()